<a href="https://colab.research.google.com/github/RDRamosU/cr-steam-comparativa-regional/blob/main/notebooks/03_limpieza_preparacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto 4 — Costa Rica vs Latinoamérica en STEAM
## Notebook 03 — Limpieza y preparación de datos

**Autor:** Ruben Dario Ramos Ulate  
**Fecha:** Junio 2026  

---

In [1]:
import pandas as pd
import numpy as np
from google.colab import files

print("Librerías importadas ✓")

Librerías importadas ✓


## 1. Carga y reconstrucción de datasets UNESCO

In [2]:
# Subir archivos UNESCO
print("Seleccionar los dos archivos CSV de UNESCO...")
uploaded = files.upload()
archivos = list(uploaded.keys())

archivo_stem = [f for f in archivos if 'STEM' in f.upper()][0]
archivo_ict  = [f for f in archivos if 'ICT' in f.upper()][0]

df_stem_raw = pd.read_csv(archivo_stem, skiprows=1)
df_stem_raw.columns = pd.read_csv(archivo_stem, nrows=0).columns

df_ict_raw = pd.read_csv(archivo_ict, skiprows=1)
df_ict_raw.columns = pd.read_csv(archivo_ict, nrows=0).columns

print(f"STEM: {df_stem_raw.shape} | ICT: {df_ict_raw.shape} ✓")

Seleccionar los dos archivos CSV de UNESCO...


Saving UNESCO_UIS_GRAD_STEM.csv to UNESCO_UIS_GRAD_STEM.csv
Saving UNESCO_UIS_GRAD_ICT.csv to UNESCO_UIS_GRAD_ICT.csv
STEM: (6690, 39) | ICT: (6743, 39) ✓


In [3]:
# Función de extracción LAC
paises_lac = [
    'Costa Rica', 'Chile', 'Argentina', 'Colombia',
    'Mexico', 'Brazil', 'Peru', 'Uruguay',
    'Panama', 'Ecuador', 'Honduras', 'Guatemala'
]

def extraer_lac(df_raw):
    df = df_raw.rename(columns={
        'REF_AREA_LABEL': 'pais',
        'TIME_PERIOD': 'año',
        'OBS_VALUE': 'valor',
        'SEX_LABEL': 'sexo',
        'UNIT_MEASURE_LABEL': 'unidad'
    })
    return df[
        (df['pais'].isin(paises_lac)) &
        (df['unidad'] == 'Percentage of graduates')
    ][['pais', 'año', 'sexo', 'valor']].dropna()

df_stem = extraer_lac(df_stem_raw)
df_ict  = extraer_lac(df_ict_raw)

print(f"STEM filtrado: {df_stem.shape} ✓")
print(f"ICT filtrado:  {df_ict.shape} ✓")

STEM filtrado: (419, 4) ✓
ICT filtrado:  (419, 4) ✓


## 2. Dataset 1 — % graduados STEM por país (dato más reciente)

In [4]:
# STEM total — dato más reciente por país
df_stem_total = df_stem[df_stem['sexo'] == 'Total']
df_stem_reciente = (df_stem_total
    .sort_values('año', ascending=False)
    .groupby('pais')
    .first()
    .reset_index()
    .sort_values('valor', ascending=False)
    .reset_index(drop=True)
)
df_stem_reciente['ranking'] = df_stem_reciente.index + 1
df_stem_reciente['es_cr'] = df_stem_reciente['pais'] == 'Costa Rica'
df_stem_reciente['promedio_lac'] = df_stem_reciente['valor'].mean().round(2)
df_stem_reciente['diferencia_promedio'] = (
    df_stem_reciente['valor'] - df_stem_reciente['promedio_lac']
).round(2)

print("=== DATASET 1 — STEM POR PAÍS ===\n")
print(df_stem_reciente[[
    'ranking', 'pais', 'año', 'valor',
    'diferencia_promedio'
]].to_string(index=False))
print(f"\nPromedio LAC: {df_stem_reciente['promedio_lac'].iloc[0]:.2f}%")

=== DATASET 1 — STEM POR PAÍS ===

 ranking       pais  año     valor  diferencia_promedio
       1       Peru 2017 29.642820                11.45
       2   Colombia 2021 23.911249                 5.72
       3     Mexico 2022 23.748711                 5.56
       4      Chile 2022 21.382799                 3.19
       5    Ecuador 2022 19.666300                 1.48
       6     Brazil 2022 16.266979                -1.92
       7 Costa Rica 2023 15.775070                -2.41
       8   Honduras 2019 15.732690                -2.46
       9  Argentina 2022 14.809300                -3.38
      10    Uruguay 2022 14.540020                -3.65
      11     Panama 2022 13.020700                -5.17
      12  Guatemala 2015  9.765440                -8.42

Promedio LAC: 18.19%


## 3. Dataset 2 — Brecha de género STEM por país

In [5]:
# Brecha de género
df_stem_genero = df_stem[df_stem['sexo'].isin(['Female', 'Male'])]
df_stem_genero_rec = (df_stem_genero
    .sort_values('año', ascending=False)
    .groupby(['pais', 'sexo'])
    .first()
    .reset_index()
)

df_genero = df_stem_genero_rec.pivot(
    index='pais', columns='sexo', values='valor'
).reset_index()
df_genero.columns.name = None
df_genero['brecha_genero'] = (
    df_genero['Male'] - df_genero['Female']
).round(2)
df_genero = df_genero.sort_values('brecha_genero').reset_index(drop=True)
df_genero['ranking_genero'] = df_genero.index + 1
df_genero['promedio_brecha_lac'] = df_genero['brecha_genero'].mean().round(2)

print("=== DATASET 2 — BRECHA DE GÉNERO STEM ===\n")
print(df_genero[[
    'ranking_genero', 'pais', 'Female', 'Male', 'brecha_genero'
]].to_string(index=False))
print(f"\nPromedio brecha LAC: {df_genero['promedio_brecha_lac'].iloc[0]:.2f} pp")

=== DATASET 2 — BRECHA DE GÉNERO STEM ===

 ranking_genero       pais   Female      Male  brecha_genero
              1  Guatemala  5.43075 16.953430          11.52
              2    Uruguay 10.39879 22.504850          12.11
              3       Peru 24.44179 36.820339          12.38
              4  Argentina 10.58690 23.704281          13.12
              5     Panama  8.17287 22.593090          14.42
              6 Costa Rica  9.38601 25.239950          15.85
              7   Honduras  9.40582 26.133801          16.73
              8     Brazil  8.56560 28.077480          19.51
              9   Colombia 15.08117 35.434540          20.35
             10    Ecuador 10.41161 31.427111          21.02
             11     Mexico 14.25955 35.865749          21.61
             12      Chile  7.80348 39.653912          31.85

Promedio brecha LAC: 17.54 pp


## 4. Dataset 3 — % graduados ICT por país

In [6]:
# ICT total — dato más reciente
df_ict_total = df_ict[df_ict['sexo'] == 'Total']
df_ict_reciente = (df_ict_total
    .sort_values('año', ascending=False)
    .groupby('pais')
    .first()
    .reset_index()
    .sort_values('valor', ascending=False)
    .reset_index(drop=True)
)
df_ict_reciente['ranking_ict'] = df_ict_reciente.index + 1
df_ict_reciente['promedio_lac_ict'] = df_ict_reciente['valor'].mean().round(2)

print("=== DATASET 3 — ICT POR PAÍS ===\n")
print(df_ict_reciente[[
    'ranking_ict', 'pais', 'año', 'valor'
]].to_string(index=False))
print(f"\nPromedio LAC ICT: {df_ict_reciente['promedio_lac_ict'].iloc[0]:.2f}%")

=== DATASET 3 — ICT POR PAÍS ===

 ranking_ict       pais  año   valor
           1       Peru 2017 5.79122
           2 Costa Rica 2023 5.48845
           3     Brazil 2022 4.62050
           4     Mexico 2022 4.23768
           5    Uruguay 2022 4.07174
           6   Colombia 2021 3.64831
           7     Panama 2022 3.51453
           8   Honduras 2019 3.43449
           9      Chile 2022 3.18680
          10    Ecuador 2022 2.43059
          11  Argentina 2022 1.65571
          12  Guatemala 2015 1.43705

Promedio LAC ICT: 3.63%


## 5. Dataset 4 — Tendencia STEM Costa Rica 2015–2023

In [7]:
# Tendencia histórica CR vs promedio LAC
años_analisis = list(range(2015, 2024))

df_tend = df_stem_total[df_stem_total['año'].isin(años_analisis)]

# Promedio LAC por año
df_prom_lac = (df_tend
    .groupby('año')['valor']
    .mean()
    .reset_index()
    .rename(columns={'valor': 'promedio_lac'})
)

# CR por año
df_cr_tend = df_tend[df_tend['pais'] == 'Costa Rica'][
    ['año', 'valor']
].rename(columns={'valor': 'costa_rica'})

# Merge
df_tendencia = df_prom_lac.merge(df_cr_tend, on='año', how='left')
df_tendencia['diferencia'] = (
    df_tendencia['costa_rica'] - df_tendencia['promedio_lac']
).round(2)

print("=== DATASET 4 — TENDENCIA CR VS PROMEDIO LAC ===\n")
print(df_tendencia.to_string(index=False))

=== DATASET 4 — TENDENCIA CR VS PROMEDIO LAC ===

 año  promedio_lac  costa_rica  diferencia
2015     17.481832    12.87717       -4.60
2016     18.796215    13.09073       -5.71
2017     20.452872    14.39941       -6.05
2018     17.187602    15.45587       -1.73
2019     18.927109    15.11316       -3.81
2020     18.432979    16.22776       -2.21
2021     19.045311    15.71689       -3.33
2022     17.401235    15.77507       -1.63
2023     15.775070    15.77507        0.00


## 5. Dataset 5 — Indicadores CR integrados con proyectos anteriores

In [8]:
# Dataset integrado CR — combina UNESCO con hallazgos P1, P2, P3
df_cr_integrado = pd.DataFrame({
    'indicador': [
        '% graduados STEM total (UNESCO)',
        '% graduados STEAM estatales (OPES-CONARE)',
        '% mujeres en graduados STEM (UNESCO)',
        '% mujeres en graduados STEAM estatales (OPES-CONARE)',
        'Tasa desempleo STEM',
        'Tasa desempleo nacional',
        'Ratio empleos ZF servicios vs graduados STEAM',
        'Salario ZF vs salario nacional',
        '% exportaciones servicios TIC del PIB',
        'Crecimiento empleo TIC anual'
    ],
    'valor_cr': [
        15.78, 34.1, 9.39, 47.3,
        4.2, 12.2, 15.0, 2.0,
        8.0, 37.3
    ],
    'referencia_lac': [
        18.19, None, None, 15.0,
        None, None, None, None,
        None, None
    ],
    'unidad': [
        '%', '%', '%', '%',
        '%', '%', 'x veces', 'x veces',
        '%', '%'
    ],
    'año': [
        2023, 2022, 2023, 2022,
        2022, 2022, 2022, 2024,
        2022, 2021
    ],
    'fuente': [
        'UNESCO UIS', 'OPES-CONARE P1',
        'UNESCO UIS', 'OPES-CONARE P1',
        'CONARE Radiografía P2', 'INEC ECE P2',
        'PROCOMER P2', 'PROCOMER P2',
        'BCCR P2', 'MICITT P2'
    ],
    'posicion_lac': [
        '7° de 12', None, '5° de 12', 'Superior al promedio',
        'Muy favorable', 'Alta', 'Sin comparable', 'Sin comparable',
        'Destacado', 'Acelerado'
    ]
})

print("=== DATASET 5 — INDICADORES CR INTEGRADOS ===\n")
print(df_cr_integrado.to_string(index=False))

=== DATASET 5 — INDICADORES CR INTEGRADOS ===

                                           indicador  valor_cr  referencia_lac  unidad  año                fuente         posicion_lac
                     % graduados STEM total (UNESCO)     15.78           18.19       % 2023            UNESCO UIS             7° de 12
           % graduados STEAM estatales (OPES-CONARE)     34.10             NaN       % 2022        OPES-CONARE P1                 None
                % mujeres en graduados STEM (UNESCO)      9.39             NaN       % 2023            UNESCO UIS             5° de 12
% mujeres en graduados STEAM estatales (OPES-CONARE)     47.30           15.00       % 2022        OPES-CONARE P1 Superior al promedio
                                 Tasa desempleo STEM      4.20             NaN       % 2022 CONARE Radiografía P2        Muy favorable
                             Tasa desempleo nacional     12.20             NaN       % 2022           INEC ECE P2                 Alta
       R

## 6. Verificación de calidad

In [9]:
print("=== VERIFICACIÓN DE CALIDAD ===\n")

datasets = [
    ('STEM por país', df_stem_reciente),
    ('Brecha género', df_genero),
    ('ICT por país', df_ict_reciente),
    ('Tendencia CR vs LAC', df_tendencia),
    ('CR integrado', df_cr_integrado)
]

for nombre, df in datasets:
    nulos = df.isnull().sum().sum()
    print(f"{nombre}:")
    print(f"  Filas: {len(df)} | Nulos: {nulos} "
          f"{'(esperados)' if nulos > 0 else '✓'}")

print("\n=== HALLAZGOS CLAVE DE LIMPIEZA ===")
cr_ranking = df_stem_reciente[
    df_stem_reciente['pais']=='Costa Rica']['ranking'].values[0]
cr_brecha  = df_genero[
    df_genero['pais']=='Costa Rica']['ranking_genero'].values[0]
cr_ict_rank = df_ict_reciente[
    df_ict_reciente['pais']=='Costa Rica']['ranking_ict'].values[0]

print(f"  Ranking CR en % STEM total:    {cr_ranking}° de 12")
print(f"  Ranking CR brecha género STEM: {cr_brecha}° (menor brecha = mejor)")
print(f"  Ranking CR en % ICT:           {cr_ict_rank}° de 12")

=== VERIFICACIÓN DE CALIDAD ===

STEM por país:
  Filas: 12 | Nulos: 0 ✓
Brecha género:
  Filas: 12 | Nulos: 0 ✓
ICT por país:
  Filas: 12 | Nulos: 0 ✓
Tendencia CR vs LAC:
  Filas: 9 | Nulos: 0 ✓
CR integrado:
  Filas: 10 | Nulos: 9 (esperados)

=== HALLAZGOS CLAVE DE LIMPIEZA ===
  Ranking CR en % STEM total:    7° de 12
  Ranking CR brecha género STEM: 6° (menor brecha = mejor)
  Ranking CR en % ICT:           2° de 12


In [10]:
# Exportar todos los datasets
df_stem_reciente.to_csv("stem_lac_reciente.csv", index=False)
df_genero.to_csv("stem_genero_lac.csv", index=False)
df_ict_reciente.to_csv("ict_lac_reciente.csv", index=False)
df_tendencia.to_csv("stem_tendencia_cr_lac.csv", index=False)
df_cr_integrado.to_csv("cr_indicadores_integrados.csv", index=False)

print("=== DATASETS EXPORTADOS ===")
print(f"  stem_lac_reciente.csv         → {len(df_stem_reciente)} registros ✓")
print(f"  stem_genero_lac.csv           → {len(df_genero)} registros ✓")
print(f"  ict_lac_reciente.csv          → {len(df_ict_reciente)} registros ✓")
print(f"  stem_tendencia_cr_lac.csv     → {len(df_tendencia)} registros ✓")
print(f"  cr_indicadores_integrados.csv → {len(df_cr_integrado)} registros ✓")
print(f"\nSiguiente paso: Notebook 04 — Visualizaciones y hallazgos")

=== DATASETS EXPORTADOS ===
  stem_lac_reciente.csv         → 12 registros ✓
  stem_genero_lac.csv           → 12 registros ✓
  ict_lac_reciente.csv          → 12 registros ✓
  stem_tendencia_cr_lac.csv     → 9 registros ✓
  cr_indicadores_integrados.csv → 10 registros ✓

Siguiente paso: Notebook 04 — Visualizaciones y hallazgos
